# Count Spatial Model Estimation

This notebook demonstrates **cross-sectional spatial model estimation** in `bayespecon` for two outcome types:

1. **Negative Binomial (NegBin)** — overdispersed count outcomes, using either NUTS (via PyMC) or a Pólya–Gamma Gibbs sampler


Both model families use structural-form parameterisations and produce `arviz.InferenceData` output compatible with all downstream diagnostics.

## When to Use NUTS vs Pólya–Gamma Gibbs

| Criterion | NUTS (`SARNegBin`) | PG-Gibbs (`SARNegBinStructural`) |
|---|---|---|
| **Model form** | Reduced: $\eta = (I - \rho W)^{-1} X\beta$ | Structural: $\eta = \rho W \eta + X\beta + \nu$ |
| **Spatial Jacobian** | Required: $\log|I - \rho W|$ added as Potential | Not needed (PG augmentation avoids it) |
| **Speed** | Slow for large $n$ (gradient through sparse solve) | Fast (conjugate blocks for $\beta$, $\sigma^2$, $\eta$; slice for $\rho$) |
| **ESS/s** | Low for spatial models | High (conjugate blocks mix well) |
| **Posterior params** | rho, beta, alpha (no σ) | rho, beta, **sigma**, alpha |
| **σ in model?** | No (reduced form) | Yes (structural noise) |
| **PyMC model graph** | ✅ Full access | ❌ Bypasses PyMC entirely |
| **Custom inference** | ✅ `pm.sample_prior_predictive`, etc. | ❌ Not available |
| **Warm start** | ❌ PyMC default init | ✅ GLM-based initialization |
**Rule of thumb**: Use `SARNegBinStructural` for $n > 500$ where NUTS ESS/s is poor. Use `SARNegBin` for small $n$ or when you need the full PyMC model graph.
**Rule of thumb**: Use `SARNegBinStructural` for $n > 500$ where NUTS ESS/s is poor. Use `SARNegBin` for small $n$ or when you need the full PyMC model graph.

In [ ]:
%load_ext autoreload
%autoreload 2

import arviz as az
import numpy as np

from bayespecon.dgp import simulate_sar_negbin
from bayespecon.models import SARNegBin, SARNegBinStructural

## Generate Data with Native DGP Function

We use `simulate_sar_negbin` from `bayespecon.dgp` to generate synthetic count data from a known SAR-NB2 process. This function supports two regimes:

- **`sigma2 = 0`** (default): Deterministic reduced form $\eta = (I - \rho W)^{-1} X\beta$, matching `SARNegBin`.
- **`sigma2 > 0`**: Structural form with Gaussian noise $\eta = \rho W \eta + X\beta + \nu$, matching `SARNegBinStructural`.

We use `sigma2 > 0` here so both model classes can recover the true parameters.

In [ ]:
# Generate SAR-NB2 data on a 10x10 rook grid (n=100)
data = simulate_sar_negbin(
    n=30,  # 30x30 grid → 900 observations
    rho=0.5,
    beta=np.array([1.0, 0.6, -0.3]),  # intercept + 2 covariates
    alpha=2.0,  # NB2 dispersion
    sigma2=0,  # reduced-form
    seed=42,
)

y = data["y"]
X = data["X"]
W = data["W_graph"]

print(f"n = {len(y)}, k = {X.shape[1]}")
print(
    f"y range: [{y.min():.0f}, {y.max():.0f}], mean = {y.mean():.1f}, var = {y.var():.1f}"
)
print(f"True params: {data['params_true']}")

## `SARNegBin`

The `SARNegBin` model uses a **reduced-form** parameterisation:

$$\eta = (I - \rho W)^{-1} X\beta$$



In [ ]:
model = SARNegBin(y=y, X=X, W=W)
idata = model.fit(draws=2000, tune=1000, chains=4, random_seed=42, gibbs_backend="jax")

print("Gibbs posterior means:")
print(f"  rho   = {float(idata.posterior['rho'].mean()):.4f}  (true = 0.5)")
print(
    f"  beta  = {idata.posterior['beta'].mean(dim=['chain', 'draw']).values}  (true = [1.0, 0.6, -0.3])"
)
print(f"  alpha = {float(idata.posterior['alpha'].mean()):.4f}  (true = 2.0)")

### with numpy

In [ ]:
model = SARNegBin(y=y, X=X, W=W)
idata = model.fit(
    draws=2000, tune=1000, chains=4, random_seed=42, gibbs_backend="numpy"
)

print("Gibbs posterior means:")
print(f"  rho   = {float(idata.posterior['rho'].mean()):.4f}  (true = 0.5)")
print(
    f"  beta  = {idata.posterior['beta'].mean(dim=['chain', 'draw']).values}  (true = [1.0, 0.6, -0.3])"
)
print(f"  alpha = {float(idata.posterior['alpha'].mean()):.4f}  (true = 2.0)")

## Pólya–Gamma Gibbs: `SARNegBinStructural`

The `SARNegBinStructural` model uses a **structural-form** parameterisation with Pólya–Gamma data augmentation:

$$y_i \sim \mathrm{NegBin}(\exp(\eta_i), \alpha), \quad \eta = \rho W \eta + X\beta + \nu, \quad \nu \sim N(0, \sigma^2 I)$$

The Gibbs sampler uses a **5-block** strategy:
1. **ω | η, y, α** — Pólya–Gamma auxiliary variables (conjugate)
2. **η | ω, β, σ², ρ** — Multivariate normal (conjugate, sparse precision solve)
3. **β | η, ω, σ²** — Multivariate normal (conjugate)
4. **σ² | η, β, ω** — Inverse-gamma (conjugate)
5. **ρ | η, β, σ²** — 1-D slice sampling or MALA (non-conjugate, scalar)
6. **α | η, y** — Griddy-Gibbs or slice (non-conjugate, scalar)

Only ρ and α are non-conjugate, and both are scalars.

In [ ]:
# Generate SAR-NB2 data on a 10x10 rook grid (n=100)
data = simulate_sar_negbin(
    n=40,  # 10x10 grid → 100 observations
    rho=0.5,
    beta=np.array([1.0, 0.6, -0.3]),  # intercept + 2 covariates
    alpha=2.0,  # NB2 dispersion
    sigma2=0.5,  # structural-form residual variance
    seed=42,
)

y = data["y"]
X = data["X"]
W = data["W_graph"]

print(f"n = {len(y)}, k = {X.shape[1]}")
print(
    f"y range: [{y.min():.0f}, {y.max():.0f}], mean = {y.mean():.1f}, var = {y.var():.1f}"
)
print(f"True params: {data['params_true']}")

In [ ]:
model_gibbs = SARNegBinStructural(y=y, X=X, W=W)
idata_gibbs = model_gibbs.fit(
    draws=2000,
    tune=1000,
    chains=4,
    random_seed=42,
    n_jobs=-1,
)

print("PG-Gibbs posterior means:")
print(f"  rho   = {float(idata_gibbs.posterior['rho'].mean()):.4f}  (true = 0.5)")
print(
    f"  beta  = {idata_gibbs.posterior['beta'].mean(dim=['chain', 'draw']).values}  (true = [1.0, 0.6, -0.3])"
)
print(
    f"  sigma = {float(idata_gibbs.posterior['sigma'].mean()):.4f}  (true = {np.sqrt(0.5):.4f})"
)
print(f"  alpha = {float(idata_gibbs.posterior['alpha'].mean()):.4f}  (true = 2.0)")

In [ ]:
az.summary(idata_gibbs, var_names=["rho", "beta", "sigma", "alpha"])

In [ ]:
model_gibbs_jax = SARNegBinStructural(y=y, X=X, W=W)
idata_gibbs_jax = model_gibbs_jax.fit(
    draws=2000,
    tune=1000,
    chains=4,
    random_seed=42,
    n_jobs=-1,
    gibbs_backend="jax",
)

print("JAX PG-Gibbs posterior means:")
print(f"  rho   = {float(idata_gibbs_jax.posterior['rho'].mean()):.4f}  (true = 0.5)")
print(
    f"  beta  = {idata_gibbs_jax.posterior['beta'].mean(dim=['chain', 'draw']).values}  (true = [1.0, 0.6, -0.3])"
)
print(
    f"  sigma = {float(idata_gibbs_jax.posterior['sigma'].mean()):.4f}  (true = {np.sqrt(0.5):.4f})"
)
print(f"  alpha = {float(idata_gibbs_jax.posterior['alpha'].mean()):.4f}  (true = 2.0)")

## InferenceData Compatibility and ArviZ Diagnostics

Both `SARNegBin` and `SARNegBinStructural` produce the same `az.InferenceData` output with `posterior`, `log_likelihood`, and `observed_data` groups. All ArviZ diagnostics work seamlessly.

In [ ]:
# ArviZ diagnostics work with both NUTS and Gibbs-produced InferenceData
print("Groups:", idata_gibbs.groups())
print()

# LOO cross-validation
loo = az.loo(idata_gibbs)
print(f"LOO: elpd = {loo.elpd_loo:.2f}, SE = {loo.se:.2f}")
print()

# WAIC
waic = az.waic(idata_gibbs)
print(f"WAIC: elpd = {waic.elpd_waic:.2f}, SE = {waic.se:.2f}")
print()

# Summary
az.summary(idata_gibbs, var_names=["rho", "sigma", "alpha"])

In [ ]:
az.plot_trace(idata_gibbs, var_names=["rho", "sigma", "alpha"])

## Overdispersion Parameter Diagnostics

The NB2 dispersion parameter $\alpha$ controls the variance-mean relationship:

$$\mathrm{Var}(y_i) = \mu_i + \mu_i^2 / \alpha$$

- **$\alpha \to \infty$**: Poisson limit (no overdispersion)
- **Small $\alpha$**: Strong overdispersion

Monitoring the posterior of $\alpha$ helps assess whether the NB model is necessary or a Poisson model would suffice.

In [ ]:
# Posterior summary for alpha
alpha_samples = idata_gibbs.posterior["alpha"].values.flatten()
print(
    f"alpha posterior: mean = {alpha_samples.mean():.3f}, "
    f"median = {np.median(alpha_samples):.3f}, "
    f"95% CI = [{np.percentile(alpha_samples, 2.5):.3f}, {np.percentile(alpha_samples, 97.5):.3f}]"
)
print("True alpha = 2.0")
print()

# Compare observed variance-mean ratio with what NB2 predicts
mu_hat = np.exp(
    np.clip(
        np.mean(idata_gibbs.posterior["eta_norm"].values, axis=(0, 1))
        if "eta_norm" in idata_gibbs.posterior
        else X @ idata_gibbs.posterior["beta"].mean(dim=["chain", "draw"]).values,
        -30,
        30,
    )
)
print(f"Observed: mean(y) = {y.mean():.1f}, var(y) = {y.var():.1f}")
print(f"Variance-mean ratio: {y.var() / y.mean():.2f}")
print(
    f"NB2 predicted ratio at posterior mean alpha: 1 + mean(y)/alpha = {1 + y.mean() / alpha_samples.mean():.2f}"
)

## Summary

The negative binomial and spatial logit model estimation in `bayespecon` provides:

### NegBin Models

1. **Two model classes**: `SARNegBin` (NUTS, non-centred parameterisation) and `SARNegBinStructural` (Pólya–Gamma Gibbs, structural form)
2. **Same output format**: Both produce `az.InferenceData` with `posterior`, `log_likelihood`, `observed_data`
3. **Multiple Gibbs backends**: CHOLMOD (fastest), SPLU (fallback), JAX dense (full JIT)
4. **Full ArviZ compatibility**: LOO, WAIC, summary, plot
5. **Full diagnostics compatibility**: Bayesian LM tests, spatial diagnostics decision tree
6. **Native DGP**: `simulate_sar_negbin` generates data matching either model specification
